# Поездки Брежнева — записи секретариата (1965–1982)

Том 2 рабочих и дневниковых записей. Данные: [`trips_segments.csv`](https://drive.google.com/file/d/16NyoAdc-KGnpYZfBr-CLXO8wZweOYAY0/view?usp=sharing) на Google Drive (или локальный файл после `python extract_trips.py`).

In [ ]:
!pip install -q plotly

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

PLOT_W = 1100
YEARS = list(range(1965, 1983))
CAT_STYLE = {
    "ussr": {"label": "СССР", "color": "#1e4d8c"},
    "foreign": {"label": "зарубеж", "color": "#a61c2e"},
    "unrec": {"label": "нераспознанные", "color": "#6b7280"},
}

DACHA_CITIES = {
    "Завидово", "Ялта", "Крым", "Симферополь", "Сочи", "Барвиха",
    "Кисловодск", "Пицунда", "Гагра", "Феодосия", "Заречье", "Астрахань",
}
REST_RE = re.compile(r"отдых|отпуск|санатор", re.I)
BAR_PALETTE = [
    "#1e4d8c", "#a61c2e", "#2d6a4f", "#b5651d", "#6b4c9a", "#2a9d8f",
    "#e76f51", "#264653", "#e9c46a", "#f4a261", "#577590", "#43aa8b",
    "#f94144", "#9b5de5", "#00bbf9",
]

CITY_TO_ISO = {
    "Завидово": "RUS", "Ленинград": "RUS", "Барвиха": "RUS", "Иркутск": "RUS",
    "Красноярск": "RUS", "Новосибирск": "RUS", "Омск": "RUS", "Саратов": "RUS",
    "Краснодар": "RUS", "Сочи": "RUS", "Кисловодск": "RUS", "Заречье": "RUS",
    "Крым": "RUS", "Ялта": "RUS", "Симферополь": "RUS", "Пицунда": "GEO", "Гагра": "GEO",
    "Киев": "UKR", "Харьков": "UKR", "Днепропетровск": "UKR",
    "Минск": "BLR", "Молдавия": "MDA", "Кишинёв": "MDA",
    "Тбилиси": "GEO", "Ереван": "ARM", "Баку": "AZE",
    "Алма-Ата": "KAZ", "Казахстан": "KAZ",
    "Ташкент": "UZB", "Узбекистан": "UZB",
    "Фрунзе": "KGZ", "Душанбе": "TJK",
    "Варшава": "POL", "Беловежская пуща": "POL",
    "Прага": "CZE", "Карловы Вары": "CZE", "Братислава": "SVK",
    "Будапешт": "HUN", "Берлин": "DEU", "Бонн": "DEU", "Франкфурт": "DEU",
    "София": "BGR", "Бухарест": "ROU",
    "Белград": "SRB", "Загреб": "HRV", "Любляна": "SVN",
    "Хельсинки": "FIN", "Париж": "FRA", "Лондон": "GBR",
    "Вашингтон": "USA", "Нью-Йорк": "USA", "Оттава": "CAN",
    "Пекин": "CHN", "Токио": "JPN", "Дели": "IND",
    "Гавана": "CUB", "Улан-Батор": "MNG", "Вена": "AUT",
    "Рим": "ITA", "Мадрид": "ESP", "Тегеран": "IRN",
}

LEGACY_COUNTRY_TO_ISO = {
    "Польша": "POL", "Чехословакия": "CZE", "Венгрия": "HUN",
    "ГДР": "DEU", "ФРГ": "DEU", "Болгария": "BGR", "Румыния": "ROU",
    "Югославия": "SRB", "Франция": "FRA", "Финляндия": "FIN",
    "Индия": "IND", "Монголия": "MNG", "США": "USA", "Куба": "CUB",
    "Иран": "IRN", "Австрия": "AUT", "СССР": "RUS",
}

ISO_NAMES = {
    "RUS": "Россия", "UKR": "Украина", "BLR": "Беларусь", "KAZ": "Казахстан",
    "UZB": "Узбекистан", "GEO": "Грузия", "ARM": "Армения", "AZE": "Азербайджан",
    "MDA": "Молдова", "KGZ": "Киргизия", "TJK": "Таджикистан",
    "POL": "Польша", "CZE": "Чехия", "SVK": "Словакия", "HUN": "Венгрия",
    "DEU": "Германия", "BGR": "Болгария", "ROU": "Румыния",
    "SRB": "Сербия", "HRV": "Хорватия", "SVN": "Словения",
    "FRA": "Франция", "GBR": "Великобритания", "FIN": "Финляндия",
    "IND": "Индия", "MNG": "Монголия", "USA": "США", "CUB": "Куба",
    "IRN": "Иран", "AUT": "Австрия", "CHN": "Китай", "JPN": "Япония",
    "ITA": "Италия", "ESP": "Испания", "CAN": "Канада",
}

UNREC_CITY_PREFIX = {
    "ялт": "RUS", "крым": "RUS", "зареч": "RUS", "заре": "RUS",
    "астрах": "RUS", "екатерин": "RUS", "красн": "RUS", "ленгор": "RUS",
    "симфер": "RUS", "сочи": "RUS", "барвих": "RUS", "кислов": "RUS",
}


def trip_modern_iso(row):
    city = row["city"] if pd.notna(row["city"]) else None
    country = row["country"] if pd.notna(row["country"]) else None

    if city and city in CITY_TO_ISO:
        return CITY_TO_ISO[city]
    if country and country in LEGACY_COUNTRY_TO_ISO:
        return LEGACY_COUNTRY_TO_ISO[country]
    if city:
        low = city.lower()
        for prefix, iso in UNREC_CITY_PREFIX.items():
            if low.startswith(prefix):
                return iso
    return None


def year_counts(frame, category):
    sub = frame[frame["category"] == category]
    return sub.groupby("year").size().reindex(YEARS, fill_value=0)


def year_counts_unrec(frame):
    un = frame[frame["category"] == "нераспознанно"]
    return un.groupby("year").size().reindex(YEARS, fill_value=0)


def bar_colors(n):
    return [BAR_PALETTE[i % len(BAR_PALETTE)] for i in range(n)]


def recognition_counts(frame):
    country_only = (
        frame["country"].notna()
        & frame["city"].isna()
        & (frame["category"] != "нераспознанно")
    )
    recognized = (frame["category"] != "нераспознанно") & frame["city"].notna()
    unrecognized = frame["category"] == "нераспознанно"
    return {
        "распознанные": int(recognized.sum()),
        "только страна": int(country_only.sum()),
        "нераспознанные": int(unrecognized.sum()),
    }


def dacha_place(row):
    if pd.notna(row["city"]) and row["city"] in DACHA_CITIES:
        return row["city"]
    if REST_RE.search(str(row["line"])):
        return "отдых (место не указано)"
    return None


def is_dacha_rest(row):
    if pd.notna(row["city"]) and row["city"] in DACHA_CITIES:
        return True
    return bool(REST_RE.search(str(row["line"])))


def layout(title, height=480):
    return dict(
        title=title,
        template="plotly_white",
        width=PLOT_W,
        height=height,
        margin=dict(l=55, r=25, t=55, b=55),
    )


DRIVE_FILE_ID = "16NyoAdc-KGnpYZfBr-CLXO8wZweOYAY0"
DRIVE_CSV_URL = f"https://drive.google.com/uc?export=download&id={DRIVE_FILE_ID}"
DRIVE_VIEW_URL = f"https://drive.google.com/file/d/{DRIVE_FILE_ID}/view"


def _local_csv_paths():
    here = Path.cwd().resolve()
    return [here / "trips_segments.csv", here.parent / "trips_segments.csv"]


def read_trips_csv():
    for path in _local_csv_paths():
        if path.is_file():
            return pd.read_csv(path, keep_default_na=False), path
    return pd.read_csv(DRIVE_CSV_URL, keep_default_na=False), DRIVE_VIEW_URL


def prepare_trips_frame(frame):
    out = frame.copy()
    for col in ("country", "city"):
        out[col] = out[col].replace("None", pd.NA)
    out["year"] = out["year"].astype(int)
    return out


raw_df, csv_source = read_trips_csv()
df = prepare_trips_frame(raw_df)
len(df), csv_source


In [ ]:
fig = go.Figure()
for cat in ("ussr", "foreign"):
    style = CAT_STYLE[cat]
    counts = year_counts(df, cat)
    fig.add_trace(
        go.Scatter(
            x=counts.index,
            y=counts.values,
            mode="lines+markers",
            name=style["label"],
            line=dict(color=style["color"], width=2),
            marker=dict(size=6),
            hovertemplate="%{x}<br>%{y}<extra>" + style["label"] + "</extra>",
        )
    )

style = CAT_STYLE["unrec"]
counts = year_counts_unrec(df)
fig.add_trace(
    go.Scatter(
        x=counts.index,
        y=counts.values,
        mode="lines+markers",
        name=style["label"],
        line=dict(color=style["color"], width=2, dash="dot"),
        marker=dict(size=5),
        hovertemplate="%{x}<br>%{y}<extra>" + style["label"] + "</extra>",
    )
)

fig.update_layout(
    template="plotly_white",
    width=PLOT_W,
    height=540,
    margin=dict(l=55, r=25, t=40, b=90),
    title=dict(text="Динамика поездок по годам", x=0.5, xanchor="center"),
    xaxis_title="год",
    yaxis_title="записей",
    yaxis_rangemode="tozero",
    legend=dict(orientation="h", yanchor="top", y=-0.18, x=0.5, xanchor="center"),
)
fig.show(config={"responsive": False})

In [ ]:
rec = recognition_counts(df)
labels = list(rec.keys())
values = list(rec.values())

fig = go.Figure(
    go.Bar(
        x=labels,
        y=values,
        marker=dict(color=["#1e4d8c", "#b5651d", "#6b7280"]),
        hovertemplate="%{x}<br>%{y}<extra></extra>",
    )
)
fig.update_layout(
    **layout("Распознавание мест"),
    xaxis_title="",
    yaxis_title="записей",
    yaxis_rangemode="tozero",
)
fig.show(config={"responsive": False})

In [ ]:
dacha = df[df.apply(is_dacha_rest, axis=1)].copy()
dacha["place"] = dacha.apply(dacha_place, axis=1)
top_dacha = dacha.groupby("place").size().sort_values(ascending=True).tail(15)

fig = go.Figure(
    go.Bar(
        x=top_dacha.values,
        y=top_dacha.index,
        orientation="h",
        marker=dict(color=bar_colors(len(top_dacha))),
        hovertemplate="%{y}<br>%{x}<extra></extra>",
    )
)
fig.update_layout(
    **layout("Топ дачных / отдыхательных поездок"),
    xaxis_title="записей",
    yaxis_title="",
)
fig.show(config={"responsive": False})

In [ ]:
foreign = df[df["category"] == "foreign"].dropna(subset=["country"])
top_countries = foreign.groupby("country").size().sort_values(ascending=True).tail(15)

fig = go.Figure(
    go.Bar(
        x=top_countries.values,
        y=top_countries.index,
        orientation="h",
        marker=dict(color=bar_colors(len(top_countries))),
        hovertemplate="%{y}<br>%{x}<extra></extra>",
    )
)
fig.update_layout(**layout("Топ посещённых стран"), xaxis_title="записей", yaxis_title="")
fig.show(config={"responsive": False})

In [ ]:
ussr = df[df["category"] == "ussr"].dropna(subset=["city"])
top_ussr = ussr.groupby("city").size().sort_values(ascending=True).tail(15)

fig = go.Figure(
    go.Bar(
        x=top_ussr.values,
        y=top_ussr.index,
        orientation="h",
        marker=dict(color=bar_colors(len(top_ussr))),
        hovertemplate="%{y}<br>%{x}<extra></extra>",
    )
)
fig.update_layout(**layout("Топ городов СССР"), xaxis_title="записей", yaxis_title="")
fig.show(config={"responsive": False})

In [ ]:
foreign_cities = foreign.dropna(subset=["city"]).copy()
foreign_cities["label"] = foreign_cities.apply(
    lambda r: r["city"] if r["city"] == r["country"] else f"{r['city']} ({r['country']})",
    axis=1,
)
top_foreign = foreign_cities.groupby("label").size().sort_values(ascending=True).tail(15)

fig = go.Figure(
    go.Bar(
        x=top_foreign.values,
        y=top_foreign.index,
        orientation="h",
        marker=dict(color=bar_colors(len(top_foreign))),
        hovertemplate="%{y}<br>%{x}<extra></extra>",
    )
)
fig.update_layout(**layout("Топ зарубежных городов"), xaxis_title="записей", yaxis_title="")
fig.show(config={"responsive": False})

In [ ]:
mapped = df.copy()
mapped["iso"] = mapped.apply(trip_modern_iso, axis=1)
by_country = (
    mapped.dropna(subset=["iso"])
    .groupby("iso", as_index=False)
    .size()
    .rename(columns={"size": "count"})
)
by_country["name"] = by_country["iso"].map(ISO_NAMES)
by_country["z_color"] = np.log1p(by_country["count"])
max_count = int(by_country["count"].max())
color_ticks = sorted({0, 1, 3, 5, 10, 20, 50, 100, max_count})

fig = go.Figure(
    go.Choropleth(
        locations=by_country["iso"],
        z=by_country["z_color"],
        customdata=by_country["count"],
        locationmode="ISO-3",
        text=by_country["name"],
        colorscale=[[0, "#e8eef5"], [0.35, "#1e4d8c"], [1, "#a61c2e"]],
        zmin=0,
        colorbar=dict(
            title="записей",
            tickvals=[np.log1p(v) for v in color_ticks],
            ticktext=[str(v) for v in color_ticks],
        ),
        marker_line_color="#9ca3af",
        marker_line_width=0.5,
        hovertemplate="%{text}<br>%{customdata} записей<extra></extra>",
    )
)
fig.update_geos(
    projection_type="natural earth",
    showland=True,
    landcolor="#f3f4f6",
    showcountries=True,
    countrycolor="#d1d5db",
    coastlinecolor="#9ca3af",
    bgcolor="white",
)
fig.update_layout(
    **layout("Все поездки на карте (современные границы)", height=620),
    geo=dict(scope="world"),
)
fig.show(config={"responsive": False})